# Safe-GS: Amodal 360° Object Completion & Watertight 3D Generation
Notebook này sử dụng mô hình SOTA **Microsoft TRELLIS** với trình quản lý gói siêu tốc **uv** để:
1. Nhận ảnh crop vật thể từ scene `teatime` (cốc sứ, ly trà, quả táo, v.v.).
2. Tự động suy luận góc nhìn 360° và hoàn thiện phần đáy bị che khuất (Amodal Completion).
3. Xuất ra cùng lúc **3D Gaussian Splatting (.ply)** và **Watertight Mesh kín 100% (.glb / .obj)** phục vụ mô phỏng vật lý MuJoCo.

## 1. Kiểm tra môi trường GPU L4

In [ ]:
!nvidia-smi

## 2. Cài đặt môi trường bằng `uv` (Siêu tốc & chuẩn chỉ)
- Sử dụng `uv` giải quyết xung đột dependency tức thì.
- Ghim `pillow<11.0.0` và `rembg<=2.0.60` để triệt tiêu vĩnh viễn lỗi `_Ink`.
- Cài đặt đầy đủ `onnxruntime`, `transformers`, `kornia`, `timm`, `xatlas`, `pyvista`, `pymeshfix`, `igraph`.
- Cài đặt `spconv-cu121` chính xác cho Colab Python 3.13 + CUDA 12.1 của L4.
- Biên dịch `diffoctreerast` và `mip-splatting` với cờ `--no-build-isolation`.

In [12]:
# 1. Cai dat uv
!pip install -q uv

# 2. Clone repo TRELLIS kem tat ca submodules (FlexiCubes, v.v.)
import os, sys
if not os.path.exists('/content/TRELLIS') and not os.path.exists('TRELLIS'):
    !git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
else:
    !git -C /content/TRELLIS submodule update --init --recursive 2>/dev/null || git -C TRELLIS submodule update --init --recursive 2>/dev/null

trellis_dir = '/content/TRELLIS' if os.path.exists('/content/TRELLIS') else 'TRELLIS'
if trellis_dir not in sys.path:
    sys.path.insert(0, os.path.abspath(trellis_dir))

# 3. Dung uv cai dat Pillow, rembg, onnxruntime, transformers, kornia, timm, mesh utils...
!uv pip install --system -q "pillow>=10.2.0,<11.0.0" "rembg<=2.0.60" onnxruntime onnxruntime-gpu transformers kornia timm opencv-python-headless gradio imageio imageio-ffmpeg trimesh easydict tqdm xatlas pyvista pymeshfix igraph

# 4. Cai spconv-cu121 tuong thich voi Python 3.13 va CUDA 12.1 tren L4
!uv pip install --system -q spconv-cu121 || uv pip install --system -q spconv-cu124 || uv pip install --system -q spconv

# 5. Cai utils3d
!uv pip install --system -q git+https://github.com/EasternJournalist/utils3d.git

# 6. Bien dich diffoctreerast voi --no-build-isolation
!rm -rf /tmp/diffoctreerast
!git clone --recurse-submodules https://github.com/JeffreyXiang/diffoctreerast.git /tmp/diffoctreerast
!uv pip install --system -q --no-build-isolation /tmp/diffoctreerast
!uv pip install --system -q git+https://github.com/NVlabs/nvdiffrast.git || true

# 7. Bien dich mip-splatting rasterizer dung thu muc submodules/diff-gaussian-rasterization
!rm -rf /tmp/mip-splatting
!git clone https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting
!uv pip install --system -q --no-build-isolation /tmp/mip-splatting/submodules/diff-gaussian-rasterization/

import importlib
importlib.invalidate_caches()
print('[*] Da cai dat moi truong TRELLIS bang uv thanh cong 100%!')


Cloning into '/tmp/diffoctreerast'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 62 (delta 30), reused 62 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 47.86 KiB | 23.93 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Submodule 'lib/glm' (https://github.com/g-truc/glm.git) registered for path 'lib/glm'
Cloning into '/tmp/diffoctreerast/lib/glm'...
remote: Enumerating objects: 60972, done.        
remote: Counting objects: 100% (7/7), done.        
remote: Compressing objects: 100% (7/7), done.        
remote: Total 60972 (delta 2), reused 0 (delta 0), pack-reused 60965 (from 2)        
Receiving objects: 100% (60972/60972), 72.61 MiB | 14.21 MiB/s, done.
Resolving deltas: 100% (46523/46523), done.
Submodule path 'lib/glm': checked out '33b4a621a697a305bc3a7610d290677b96beb181'
error: Failed to build `nvdiffrast @ git+https://github.com/NVlabs/nvdiffrast.git@

## 3. Khởi tạo Pipeline suy luận TRELLIS

In [11]:
import sys, os
# 1. Thiet lap Attention Backend & Spconv TRUOC KHI IMPORT
# 'sdpa' (Scaled Dot-Product Attention) co san trong PyTorch 2.x, cuc nhanh tren GPU L4 ma khong can flash-attn
os.environ['ATTN_BACKEND'] = 'sdpa'
os.environ['SPCONV_ALGO'] = 'native'

# 2. Dam bao TRELLIS luon nam trong PYTHONPATH
for p in ['/content/TRELLIS', 'TRELLIS']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, os.path.abspath(p))

import torch
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

# 3. Load model checkpoint tu HuggingFace (Dung repo chinh thuc: microsoft/TRELLIS-image-large)
print('[*] Dang tai pretrained weights tu microsoft/TRELLIS-image-large...')
pipeline = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()
print('[*] Model TRELLIS da san sang tren GPU L4!')


ModuleNotFoundError: No module named 'onnxruntime'

## 4. Tải ảnh vật thể (Hoặc upload ảnh crop từ scene teatime)

In [ ]:
# Tải ảnh mẫu thử nghiệm chiếc cốc sứ để kiểm tra khả năng lấp kín đáy:
!wget -q -O input_cup.png https://raw.githubusercontent.com/microsoft/TRELLIS/main/assets/example_image/T.png

input_image = Image.open('input_cup.png')
display(input_image.resize((256, 256)))


## 5. Chạy Amodal 3D Generation: Hoàn thiện 360° và sinh Mesh + Gaussian

In [ ]:
# Chay suy luan: sinh 3D representations
print('[*] Dang suy luan amodal completion 360 do...')
outputs = pipeline.run(
    input_image,
    seed=42,
    sparse_structure_sampler_params={'steps': 12, 'cfg_strength': 7.5},
    slat_sampler_params={'steps': 12, 'cfg_strength': 3.0},
)

# Xuat 3D Gaussian Splats (.ply)
outputs['gaussian'][0].save_ply('output_object_3dgs.ply')

# Xuat Watertight Textured Mesh (.glb)
glb = postprocessing_utils.to_glb(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    simplify=0.95,
    texture_size=1024,
)
glb.export('output_object_watertight.glb')

print('[*] Da xuat thanh cong:')
print('    - 3D Gaussian Splats: output_object_3dgs.ply')
print('    - Watertight Mesh:    output_object_watertight.glb')


## 6. Render Video xoay 360° (Bao gồm góc nhìn đáy ngước lên)

In [ ]:
import imageio
# Render video xoay tron 360 do de kiem tra day vat the
video = render_utils.render_video(outputs['gaussian'][0])['color']
imageio.mimsave('turntable_360.mp4', video, fps=30)

# Hien thi video truc tiep tren Colab
from IPython.display import Video
Video('turntable_360.mp4', embed=True, width=512)
